# Étape 9 — Pipeline v3.2 : post-traitement déterministe

Le modèle v3 a deux faiblesses mesurées au test vierge : il **oublie des compétences** (rappel) et il **calcule mal les durées** (jours → mois, fourchettes). On ajoute deux règles après le modèle, **sans réentraîner** :

1. **Durée** : relue dans le texte de l'offre (fourchette → borne basse, jours ÷ 30, semaines × 7 ÷ 30), en ignorant l'expérience demandée, les dates de publication, la période d'essai et les rythmes hebdomadaires.
2. **Lexique** : les compétences déjà vues dans les annotations, présentes mot pour mot dans l'offre mais oubliées par le modèle, sont ajoutées. Chaque entrée a une précision mesurée ; on ne garde que les entrées assez fiables.

**Protocole (pour ne pas tricher) :**
- les règles et le lexique sont construits sur le **train** ;
- les réglages (stratégie de durée, seuil du lexique) sont choisis sur le **test de développement**, avec les sorties v3 déjà enregistrées ;
- le **test vierge** n'est utilisé qu'**une seule fois**, à la fin, avec les réglages figés.

⚠️ Le test vierge a déjà été consulté une fois (notebook 06) : son résultat ici est indicatif. Une validation définitive demandera un *test vierge 2*.

**Pas besoin de GPU** : on réutilise les sorties du modèle déjà enregistrées sur Drive. Durée : 2 minutes.

In [ ]:
# 1. Drive + code du post-traitement (dossier demo/ du repo GitHub)
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RACINE = '/content/drive/MyDrive/job-offer-extractor'
DOSSIER, RESULTATS = f'{RACINE}/data', f'{RACINE}/resultats'

%cd /content
!rm -rf job-offer-extractor && git clone -q https://github.com/mayss111/job-offer-extractor.git
sys.path.insert(0, '/content/job-offer-extractor/demo')
from post_traitement import Lexique, duree_depuis_texte, corriger_duree
from normalisation import ensemble, cle
print('post_traitement.py chargé')

In [ ]:
# 2. Schéma et métriques : le même code que les notebooks 05 et 06
import json, re
import pandas as pd
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal['PFE', 'stage', 'CDI', 'CDD', 'SIVP', 'freelance']]
    mode_travail: Optional[Literal['présentiel', 'hybride', 'remote']]
    duree_mois: Optional[float]
    remuneration: Optional[str]
    langue_offre: Literal['fr', 'en', 'ar']
    competences: list[str]

CHAMPS = ['poste', 'entreprise', 'ville', 'type_contrat', 'mode_travail',
          'duree_mois', 'remuneration', 'langue_offre']
GARDER_SLASH = {'ci/cd', 'tcp/ip', 'pl/sql', 'pub/sub', 'tests a/b', 'a/b'}

def nettoyer(texte):
    return texte.strip().removeprefix('```json').removesuffix('```').strip()

def reparer(brut):
    t = nettoyer(brut)
    i = t.find('"competences"')
    if i < 0:
        return t
    debut = t.index('[', i)
    items, vus = [], set()
    for m in re.finditer(r'"((?:[^"\\]|\\.)*)"', t[debut:]):
        try:
            item = json.loads('"' + m.group(1) + '"')
        except json.JSONDecodeError:
            break
        if item.strip().lower() in vus:
            break
        vus.add(item.strip().lower())
        items.append(item)
    return t[:debut] + json.dumps(items, ensure_ascii=False) + '}'

def norm(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return round(float(v), 2)
    return ' '.join(str(v).lower().split())

def egal(champ, pred, vrai):
    if champ == 'duree_mois' and pred is not None and vrai is not None:
        try:
            return abs(float(pred) - float(vrai)) < 0.05
        except (TypeError, ValueError):
            return False
    return norm(pred) == norm(vrai)

def norm_souple(v):
    if v is None:
        return None
    t = str(v).lower()
    t = re.sub(r'\(?\b(h/f|f/h)\b\)?', ' ', t)
    t = re.sub(r'[^\w\s#+./]', ' ', t)
    return ' '.join(t.split())

def eclater(competences):
    out = set()
    for c in competences or []:
        if not isinstance(c, str):
            continue
        n = norm_souple(c)
        if n in GARDER_SLASH:
            out.add(n)
        else:
            out.update(p.strip() for p in n.split('/') if p.strip())
    return out

def ancre(terme, texte):
    motif = r'(?<!\w)' + re.escape(terme.strip()) + r'(?!\w)'
    return re.search(motif, texte, flags=re.IGNORECASE) is not None

def scorer(nom, donnees, sorties):
    """sorties = JSON finaux (déjà réparés / post-traités), un par offre de `donnees`."""
    lignes, tp, fp, fn = [], 0, 0, 0
    stp, sfp, sfn, n_pred, n_ancre = 0, 0, 0, 0, 0
    for ex, sortie in zip(donnees, sorties):
        vrai = ex['extraction']
        ligne = {'url': ex.get('url'), 'langue': vrai['langue_offre'], 'sortie': sortie}
        try:
            pred = json.loads(sortie)
        except (json.JSONDecodeError, TypeError):
            pred = {}
        try:
            Offre.model_validate(pred)
            ligne['schema_ok'] = True
        except ValidationError:
            ligne['schema_ok'] = False
        for c in CHAMPS:
            ligne[c] = egal(c, pred.get(c), vrai.get(c))
        ligne['poste_souple'] = norm_souple(pred.get('poste')) == norm_souple(vrai.get('poste'))
        comps = [x for x in (pred.get('competences') or []) if isinstance(x, str)]
        p, v = {norm(x) for x in comps}, {norm(x) for x in vrai['competences']}
        tp += len(p & v); fp += len(p - v); fn += len(v - p)
        ps, vs = eclater(comps), eclater(vrai['competences'])
        stp += len(ps & vs); sfp += len(ps - vs); sfn += len(vs - ps)
        morceaux = [m.strip() for x in comps
                    for m in ([x] if norm_souple(x) in GARDER_SLASH else x.split('/')) if m.strip()]
        n_pred += len(morceaux)
        n_ancre += sum(ancre(m, ex['offre']) for m in morceaux)
        lignes.append(ligne)
    df = pd.DataFrame(lignes)

    def prf(a, b, c):
        pr = a / (a + b) if a + b else 0
        rp = a / (a + c) if a + c else 0
        return pr, rp, (2 * pr * rp / (pr + rp) if pr + rp else 0)

    prec, rapp, f1 = prf(tp, fp, fn)
    sprec, srapp, sf1 = prf(stp, sfp, sfn)
    resume = {'Schéma respecté': df['schema_ok'].mean()}
    resume.update({c: df[c].mean() for c in CHAMPS})
    resume['poste_souple'] = df['poste_souple'].mean()
    resume.update({'Compétences précision': prec, 'Compétences rappel': rapp, 'Compétences F1': f1,
                   'Compétences souples P': sprec, 'Compétences souples R': srapp,
                   'Compétences souples F1': sf1,
                   'Taux d’ancrage': n_ancre / n_pred if n_pred else 0,
                   'Compétences / offre': n_pred / len(df)})
    df.to_csv(f'{RESULTATS}/details_v32_{nom}.csv', index=False)
    return pd.Series(resume, name=nom).round(3)

In [ ]:
# 3. Données et sorties v3 déjà enregistrées
def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train = lire(f'{DOSSIER}/train.jsonl')
dev = lire(f'{DOSSIER}/test.jsonl')               # test de développement (14)
vierge = lire(f'{DOSSIER}/test_vierge.jsonl')     # test vierge (14)

def sorties_enregistrees(fichier, donnees):
    d = pd.read_csv(f'{RESULTATS}/{fichier}')
    assert list(d['url']) == [ex.get('url') for ex in donnees], f'Ordre différent dans {fichier}'
    return d['sortie_brute'].fillna('').tolist()

brut_dev = sorties_enregistrees('details_finetune_v3_repare.csv', dev)
brut_vierge = sorties_enregistrees('details_vierge_v3_brut.csv', vierge)
print(f'train {len(train)} | dev {len(dev)} | vierge {len(vierge)} — sorties v3 chargées')

In [ ]:
# 4. Pipeline : v3 + garde-fous (référence) et v3.2 (nouvelles règles)
DUREE_RE = re.compile(
    r'\d+([.,]\d+)?\s*(mois|month|months|semaines?|weeks?|jours?|days?|ans?|years?)'
    r'|\d+\s*(يوما|يوم|أيام|أسابيع|أشهر|شهر)|يومين|شهرين', re.IGNORECASE)

def lire_pred(brut):
    try:
        return json.loads(nettoyer(brut))
    except json.JSONDecodeError:
        try:
            return json.loads(reparer(brut))
        except Exception:
            return {}

def confidentiel(pred):
    if isinstance(pred.get('remuneration'), str) and pred['remuneration'].strip().lower() in {'confidentiel', 'confidential'}:
        pred['remuneration'] = None
    return pred

def pipeline_v3(brut, offre):
    """Pipeline publié jusqu'ici : réparation + garde-fous du notebook 06."""
    pred = lire_pred(brut)
    if not pred:
        return nettoyer(brut)
    if pred.get('duree_mois') is not None and not DUREE_RE.search(offre):
        pred['duree_mois'] = None
    return json.dumps(confidentiel(pred), ensure_ascii=False)

def pipeline_v32(brut, offre, strategie, lexique=None, seuil=None):
    pred = lire_pred(brut)
    if not pred:
        return nettoyer(brut)
    pred = corriger_duree(confidentiel(pred), offre, strategie)
    if lexique is not None and seuil is not None and isinstance(pred.get('competences'), list):
        pred['competences'], _ = lexique.completer(pred['competences'], offre, seuil=seuil)
    return json.dumps(pred, ensure_ascii=False)

# Contrôle : on doit retrouver les chiffres déjà publiés (dev F1 ≈ 0.70, vierge F1 ≈ 0.50)
ref_dev = scorer('dev_v3', dev, [pipeline_v3(b, ex['offre']) for b, ex in zip(brut_dev, dev)])
print('Contrôle dev  : F1 compétences =', ref_dev['Compétences F1'], '| durée =', ref_dev['duree_mois'])

## Règle de durée

D'abord la règle **seule**, comparée aux étiquettes du **train** (aucun modèle) : c'est là qu'on peut regarder les erreurs et corriger la règle sans toucher aux tests.

In [ ]:
# 5. La règle seule sur le train (54 offres annotées)
erreurs, ok = [], 0
for ex in train:
    vrai, regle = ex['extraction']['duree_mois'], duree_depuis_texte(ex['offre'])
    if egal('duree_mois', regle, vrai):
        ok += 1
    else:
        erreurs.append((vrai, regle, ex['extraction']['poste'][:50]))
print(f'Règle seule, train : {ok}/{len(train)} = {ok/len(train):.3f}')
for vrai, regle, poste in erreurs:
    print(f'  attendu {vrai!s:6} | règle {regle!s:6} | {poste}')

In [ ]:
# 6. Choix de la stratégie sur le test de développement (sorties v3)
STRATEGIES = ['modele', 'si_trouvee', 'stricte']
res = {}
for s in STRATEGIES:
    sorties = [pipeline_v32(b, ex['offre'], s) for b, ex in zip(brut_dev, dev)]
    res[s] = sum(egal('duree_mois', json.loads(o).get('duree_mois') if o.startswith('{') else None,
                      ex['extraction']['duree_mois']) for o, ex in zip(sorties, dev)) / len(dev)
print('Durée correcte sur le dev :', {k: round(v, 3) for k, v in res.items()})
# en cas d'égalité, on garde la plus prudente (la première de la liste)
STRATEGIE = max(STRATEGIES, key=lambda s: (res[s], -STRATEGIES.index(s)))
print('→ stratégie retenue :', STRATEGIE)

## Lexique des compétences

Construit sur le **train** uniquement. Pour chaque compétence, la *précision* = parmi les offres du train où elle apparaît dans le texte, la part où elle figure dans les étiquettes. Une précision basse signale un mot souvent cité sans être demandé (présentation de l'entreprise, tâche…).

In [ ]:
# 7. Lexique construit sur le train
lex_train = Lexique.construire(train)
stats = pd.DataFrame([{'competence': n, 'formes': ', '.join(e['formes']), 'vu': e['vu'], 'precision': e['precision']}
                      for n, e in lex_train.entrees.items()]).sort_values(['precision', 'vu'])
print(f'{len(stats)} compétences dans le lexique')
print('Les moins fiables :')
stats[stats['vu'] > 0].head(15)

In [ ]:
# 8. Choix du seuil sur le test de développement
SEUILS = [None, 0.5, 0.7, 0.8, 0.9, 1.0]
lignes = []
for seuil in SEUILS:
    sorties = [pipeline_v32(b, ex['offre'], STRATEGIE, lex_train, seuil) for b, ex in zip(brut_dev, dev)]
    r = scorer(f'dev_seuil_{seuil}', dev, sorties)
    lignes.append({'seuil': 'sans lexique' if seuil is None else seuil,
                   'précision': r['Compétences précision'], 'rappel': r['Compétences rappel'],
                   'F1': r['Compétences F1'], 'compétences / offre': r['Compétences / offre']})
choix_seuil = pd.DataFrame(lignes)
display(choix_seuil)
# meilleur F1 ; en cas d'égalité, le seuil le plus élevé (le plus prudent)
meilleur = max(range(len(SEUILS)), key=lambda i: (lignes[i]['F1'], i))
SEUIL = SEUILS[meilleur]
print('→ seuil retenu :', SEUIL if SEUIL is not None else 'sans lexique')

In [ ]:
# 9. Ce que le lexique ajoute sur le dev (vérification à l'œil)
if SEUIL is not None:
    for b, ex in zip(brut_dev, dev):
        pred = lire_pred(b)
        _, ajouts = lex_train.completer(pred.get('competences') or [], ex['offre'], seuil=SEUIL)
        if ajouts:
            etiq = ensemble(ex['extraction']['competences'])
            marques = [f"{a} {'✅' if cle(a) in etiq else '❌'}" for a in ajouts]
            print(ex['extraction']['poste'][:45].ljust(45), '→', ', '.join(marques))

## Évaluation finale (une seule fois)

Réglages **figés** ci-dessous. Pour le test vierge, le lexique est reconstruit sur **train + dev** (68 offres, toutes déjà connues) : c'est le lexique qui sera utilisé dans la démo.

In [ ]:
# 10. Réglages figés + évaluation finale
CONFIG = {'version': 'v3.2', 'strategie_duree': STRATEGIE, 'seuil_lexique': SEUIL}
print('Réglages figés :', CONFIG)

lex_final = Lexique.construire(train + dev)

tableau = pd.concat([
    ref_dev.rename('dev · v3'),
    scorer('dev_v32', dev, [pipeline_v32(b, ex['offre'], STRATEGIE, lex_train, SEUIL)
                            for b, ex in zip(brut_dev, dev)]).rename('dev · v3.2'),
    scorer('vierge_v3', vierge, [pipeline_v3(b, ex['offre'])
                                 for b, ex in zip(brut_vierge, vierge)]).rename('vierge · v3'),
    scorer('vierge_v32', vierge, [pipeline_v32(b, ex['offre'], STRATEGIE, lex_final, SEUIL)
                                  for b, ex in zip(brut_vierge, vierge)]).rename('vierge · v3.2'),
], axis=1)
tableau.to_csv(f'{RESULTATS}/comparaison_v32.csv')
print(f'1 offre = {100/len(vierge):.1f} points')
tableau

In [ ]:
# 11. Exporter le post-traitement pour la démo
sortie = {**CONFIG, 'lexique': {n: e for n, e in lex_final.entrees.items()
                                 if SEUIL is not None and e['precision'] is not None and e['precision'] >= SEUIL}}
with open(f'{RESULTATS}/post_traitement_v3_2.json', 'w', encoding='utf-8') as f:
    json.dump(sortie, f, ensure_ascii=False, indent=1, sort_keys=True)
print(f"{len(sortie['lexique'])} compétences exportées dans {RESULTATS}/post_traitement_v3_2.json")
print('→ télécharge ce fichier et mets-le dans le dossier demo/ du repo GitHub.')